In [23]:
from pydantic import BaseModel, Field
from typing import List
from langchain_groq import ChatGroq

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from tqdm import tqdm

model_name_openai = "gpt-4o-2024-08-06" # Garante 100% do output em JSON

llm_openai = ChatOpenAI(
    model=model_name_openai, # 100% json output
    temperature=0,
)

system_prompt = """
Você é um assistente inteligente que vai construir uma conversa entre duas personas. Considere que a conversa pode ser natural (pode ser como num formato de podcast). Seu trabalho e gerar uma conversa extensa que começa com tópicos relacionados à {topic_1}, mas que gradualmente se move para tópicos relacionados a {topic_2}.

REGRA: Apenas a persona 2 é quem conduz a conversa. Ele introduz a mudança do assunto como um moderador faria, fazendo perguntas ou comentários que levam a conversa para o novo tópico. A persona 1 responde às perguntas e comentários, mas não conduz a conversa.

Além disso, gere uma quantidade significativa de mensagens (pelo menos 100 trocas de mensagens) para garantir uma transição suave e gradual entre os tópicos. É preciso que o moderador faça questionamentos importantes para que a conversa seja rica e interessante.

Siga o seguinte padrão para a conversa:

Persona 1: [fala algo que pode ser relacionado a {topic_1}]
Persona 2: [responde, ainda relacionado a {topic_1} ou fazendo uma transição suave]
... 


---------------
"""

prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "Gere a conversa entre as duas personas."),
        ]
)

chain = prompt | llm_openai | StrOutputParser()

In [24]:
response = chain.invoke({
    "topic_1": "história",
    "topic_2": "educação financeira"
})

In [25]:
def select_persona(message: str) -> str:
    if message.startswith("Persona 1:"):
        return "Persona 1"
    elif message.startswith("Persona 2:"):
        return "Persona 2"
    else:
        return "Unknown"

conversation_list = response.split("\n")
conversation_list = [line for line in conversation_list if line.strip() != ""]
conversation_list_persona = [{"persona": select_persona(line), "message": line[11:]} for line in conversation_list]

In [26]:
system_prompt_summary = """
Você é um assistente inteligente que vai resumir uma conversa entre duas personas. Você vai receber uma lista de mensagens, cada uma associada a uma persona (Persona 1 ou Persona 2). Seu trabalho é gerar um resumo conciso da conversa, dê enfase para o tópico principal discutido na conversa no momento atual (mesmo que o tópico tenha mudado ao longo da conversa). Os tópicos devem girar em torno de {topic_1} e {topic_2}.
"""

prompt_summary = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_summary),
            ("human", "Resuma a conversa dada: {conversation}"),
        ]
)

chain_summary = prompt_summary | llm_openai | StrOutputParser()

In [27]:
summary_list = []
buffer = []
for i in tqdm(range(len(conversation_list))):
    buffer.append(conversation_list[i])
    buffer_trimmed = buffer[-10:]  # keep only the last 10 messages
    buffer_str = "\n".join(buffer_trimmed)
    response_summary = chain_summary.invoke({
        "topic_1": "história",
        "topic_2": "educação financeira",
        "conversation": buffer_str
    })
    summary_list.append(response_summary)

100%|██████████| 25/25 [01:16<00:00,  3.05s/it]


In [28]:
len(conversation_list), len(summary_list), len(conversation_list_persona)

(25, 25, 25)

In [29]:
all_data_list = []
for i in range(len(conversation_list)):
    all_data_list.append({
        "summary": summary_list[i],
        "persona": conversation_list_persona[i]["persona"],
        "message": conversation_list_persona[i]["message"]
    })

In [30]:
# save list of dicts to json file as a list of json objects
import json

with open("../data/conversation/conversation_data_major_3.json", "w", encoding="utf-8") as f:
    json.dump(all_data_list, f, ensure_ascii=False, indent=4)

In [31]:
# Load data from json file

with open("../data/conversation/conversation_data_major_3.json", "r", encoding="utf-8") as f:
    loaded_content = json.load(f)   
    print(loaded_content[:5])

[{'summary': 'A conversa gira em torno da Revolução Industrial e seu impacto transformador no mundo, destacando como ela alterou significativamente a maneira como vivemos e trabalhamos.', 'persona': 'Persona 1', 'message': 'Você já parou para pensar em como a Revolução Industrial transformou o mundo? É fascinante como ela mudou a forma como vivemos e trabalhamos.'}, {'summary': 'A conversa entre as duas personas gira em torno da Revolução Industrial e seu impacto transformador no mundo. Persona 1 destaca como essa era mudou a forma de viver e trabalhar, enquanto Persona 2 concorda, mencionando a transição de uma economia agrária para uma industrializada e perguntando qual aspecto Persona 1 considera mais interessante. O foco principal é a importância histórica e as inovações trazidas pela Revolução Industrial.', 'persona': 'Persona 2', 'message': 'Com certeza, a Revolução Industrial foi um ponto de virada na história. A transição de uma economia agrária para uma industrializada trouxe 

In [32]:
loaded_content[:5]

[{'summary': 'A conversa gira em torno da Revolução Industrial e seu impacto transformador no mundo, destacando como ela alterou significativamente a maneira como vivemos e trabalhamos.',
  'persona': 'Persona 1',
  'message': 'Você já parou para pensar em como a Revolução Industrial transformou o mundo? É fascinante como ela mudou a forma como vivemos e trabalhamos.'},
 {'summary': 'A conversa entre as duas personas gira em torno da Revolução Industrial e seu impacto transformador no mundo. Persona 1 destaca como essa era mudou a forma de viver e trabalhar, enquanto Persona 2 concorda, mencionando a transição de uma economia agrária para uma industrializada e perguntando qual aspecto Persona 1 considera mais interessante. O foco principal é a importância histórica e as inovações trazidas pela Revolução Industrial.',
  'persona': 'Persona 2',
  'message': 'Com certeza, a Revolução Industrial foi um ponto de virada na história. A transição de uma economia agrária para uma industrializad